In [1]:
import numpy as np
import pandas as pd
import skimage as ski
import matplotlib.pyplot as plt
import tifffile
import zarr
import os
import xml.etree.ElementTree as ET
import napari
from importlib import reload
from instanseg import InstanSeg
import load_assess_image as load_assess
import qc_widget


In [90]:
from instanseg import InstanSeg
from skimage.measure import regionprops_table
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))  # shows your GPU name

True
NVIDIA GeForce RTX 3060


In [49]:
reload(load_assess)

<module 'load_assess_image' from 'r:\\Thu\\From_2026\\6. Other\\DL_Summer_2026\\notebook\\load_assess_image.py'>

In [ ]:
main_dir=rf'D:\thu\HTAN\images\ILC\level_2'
file_name= #'OHSU_TMA1_010-D2.ome.tif' BEAUTFIFUL STAINING
image_test=load_assess.AssessImage(main_dir=main_dir,
                                    filename=file_name)

In [ ]:
# view_list=['CD45','CD44','PDPN','aSMA','panCK','HLA-DPB1'] either PDPN or Vim
view_list_2=['CD45','aSMA','panCK','Vim']

In [78]:
image_test_qc_df=load_assess.AssessImage.get_stats(image_test,resolution_level=2) # set at resolution 2, good enough for qc

In [99]:
image_test.view_images(biomarkers_of_interest=view_list_2,resolution_level=0)

Viewer(mouse_move_callbacks=[], mouse_wheel_callbacks=[<function dims_scroll at 0x00000215ACFE9080>], mouse_drag_callbacks=[<function drag_to_zoom at 0x00000215ACFE91C0>], mouse_double_click_callbacks=[<function double_click_to_zoom at 0x00000215ACFE9120>], camera=Camera(center=(0.0, 1335.5, 1335.5), zoom=0.21119011976047902, angles=(0.0, 0.0, 0.0), perspective=0.0, mouse_pan=True, mouse_zoom=True, orientation=(<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.RIGHT: 'right'>)), cursor=Cursor(position=(1.0, 1.0), viewbox=None, scaled=True, size=1.0, style=<CursorStyle.STANDARD: 'standard'>), dims=Dims(ndim=2, ndisplay=2, order=(0, 1), axis_labels=('-2', '-1'), rollable=(True, True), range=(RangeTuple(start=0.0, stop=2671.0, step=1.0), RangeTuple(start=0.0, stop=2671.0, step=1.0)), margin_left=(0.0, 0.0), margin_right=(0.0, 0.0), point=(1335.0, 1335.0), units=(<Unit('pixel')>, <Unit('pixel')>), last_used=0), grid=GridCanvas(str

In [79]:
open_image_test= zarr.open(image_test.loader.image_store,mode='r')
resolution_level=0
for biomarker,channel_index in image_test.biomarker_channel.items():
    if biomarker in view_list_2:
        img=open_image_test[str(resolution_level)][int(channel_index)]
        print(biomarker,":" ,np.max(img))

Vim : 17067
aSMA : 65535
CD45 : 49618
panCK : 65535


In [80]:
cyto = image_test.max_projection(view_list_2, resolution_level=0)

In [81]:
viewer = image_test.view_images(biomarkers_of_interest=['dna'], resolution_level=0)
viewer.add_image(cyto, name='MIP',blending='additive',colormap='magenta')


<Image layer 'MIP' at 0x21b410e3a10>

True
NVIDIA GeForce RTX 3060


In [82]:

model = InstanSeg('fluorescence_nuclei_and_cells',device='cuda')


Model fluorescence_nuclei_and_cells version 0.1.1 already downloaded in d:\conda_envs\instanseg\Lib\site-packages\instanseg\utils\../bioimageio_models/, loading


In [83]:
open_image_test=zarr.open(image_test.loader.image_store,mode='r') # zarr object with stacked biomarkers so when open, need resolution and the channel_index of the biomarker
image_dna=open_image_test[str(0)][0] # dna always 0, resolution_level only take str

In [84]:
print(image_dna.shape)
print(cyto.shape)

(2672, 2672)
(2672, 2672)


In [85]:
stack=np.stack(arrays=(image_dna,cyto))
stack.shape

(2, 2672, 2672)

In [86]:
for element in image_test.loader.root.iter():
    if element.tag.endswith('Pixels'):
        print(element.attrib)

{'ID': 'Pixels:0', 'DimensionOrder': 'XYCZT', 'Type': 'uint16', 'SizeX': '2672', 'SizeY': '2672', 'SizeC': '40', 'SizeZ': '1', 'SizeT': '1', 'PhysicalSizeX': '0.6499999761581421', 'PhysicalSizeXUnit': 'µm', 'PhysicalSizeY': '0.6499999761581421', 'PhysicalSizeYUnit': 'µm'}


In [87]:
model = InstanSeg('fluorescence_nuclei_and_cells', device='cuda')
labeled_output = model.eval_small_image(stack, pixel_size=0.65)


Model fluorescence_nuclei_and_cells version 0.1.1 already downloaded in d:\conda_envs\instanseg\Lib\site-packages\instanseg\utils\../bioimageio_models/, loading


In [88]:
print(type(labeled_output))
print(len(labeled_output))
for i, item in enumerate(labeled_output):
    print(i, type(item), item.shape if hasattr(item, 'shape') else item)


<class 'tuple'>
2
0 <class 'torch.Tensor'> torch.Size([1, 2, 2672, 2672])
1 <class 'torch.Tensor'> torch.Size([1, 2, 2672, 2672])


In [89]:
nuclei_mask = labeled_output[0][0, 0].cpu().numpy().astype(int)
cells_mask  = labeled_output[0][0, 1].cpu().numpy().astype(int)

viewer = image_test.view_images(biomarkers_of_interest=['dna'], resolution_level=0)
viewer.add_image(cyto, name='MIP')
viewer.add_labels(nuclei_mask, name='nuclei')
viewer.add_labels(cells_mask,  name='cells')


<Labels layer 'cells' at 0x216f58a5f10>

In [98]:
resolution_level = 0
open_image_test = zarr.open(image_test.loader.image_store, mode='r')

# shape props once — independent of biomarker
shape_props = regionprops_table(
    cells_mask,
    properties=['label', 'area', 'centroid', 'equivalent_diameter_area',
                'solidity', 'eccentricity', 'axis_major_length', 'axis_minor_length', 'perimeter']
)
df = pd.DataFrame(shape_props)

# intensity per biomarker — rename columns to avoid collision, merge on label
for biomarker, channel_index in image_test.biomarker_channel.items():
    image_biomarker = np.array(open_image_test[str(resolution_level)][int(channel_index)])
    intensity_props = regionprops_table(
        cells_mask,
        intensity_image=image_biomarker,
        properties=['label', 'intensity_mean', 'intensity_median', 'intensity_std']
    )
    intensity_df = pd.DataFrame(intensity_props).rename(columns={
        'intensity_mean':   f'{biomarker}_mean',
        'intensity_median': f'{biomarker}_median',
        'intensity_std':    f'{biomarker}_std',
    })
    df = df.merge(intensity_df, on='label')

output_path = os.path.join(r'r:\Thu\From_2026\6. Other\DL_Summer_2026\output',
                           file_name.replace('.ome.tif', '_cells.csv'))
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} cells → {output_path}")


Saved 9244 cells → r:\Thu\From_2026\6. Other\DL_Summer_2026\output\OHSU_TMA1_010-D2_cells.csv
